# WP1 / C1 - native-segmentation baselines + MO-WOA+ with equal weights

Runs on its own: D1-D3 are copied from MO_comparison_phaseD.ipynb (only reads from Drive), then W0-W7.
Runtime -> Run all. CPU is enough (~1-2 h, W5 is checkpointed so it picks up again after a disconnect).

The files I keep afterwards: mowoa_mo_study/wp1/wp1_summary.txt and wp1_per_article.csv.

## D1 - setup (from phase D)

In [ ]:
!pip install pymoo rouge-score -q

import os, json, pickle, time, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/mowoa_mo_study'
PHASEC_DIR = f'{DRIVE_DIR}/phaseC'
PHASED_DIR = f'{DRIVE_DIR}/phaseD'
os.makedirs(PHASED_DIR, exist_ok=True)

CACHE_PATH     = f'{DRIVE_DIR}/article_caches_150.pkl'
CANONICAL_JSON = f'{DRIVE_DIR}/mowoa_summaries_150_rc1_w32_update_test.json'

N_ARTICLES = 150
ALGOS  = ['MO-WOA+', 'NSGA-II', 'MOPSO', 'MOEA/D']     # main comparison (k=4)
ABLREM = 'MO-WOA'                                       # all-enhancements-removed

## D2 - shared module (from phase D, code unchanged)

In [ ]:
%%writefile mowoa_mo_shared.py
"""
shared layer for the MO-WOA multi-objective comparison (phase B, axis 2).

all four algorithms read the same enumerated objective table and use the same
encoding, repair, initial population per (article, seed) and FE budget, so
objective identity holds by construction.

protocol decisions: D-1 shared P0 (paired design), D-2 terminal + cumulative
fronts, D-3 B = 1480 = 30 + 50 x 29 (surplus goes to the peers, never MO-WOA),
D-4 no tuning, D-6 strata by m = |P*|: A m>=3, B m==2, C m==1 (degenerate).
summary_size=3, repair_solution(), position bonus, knee_weights=[32, 1], the
70/30 leader split and leader preservation are copied verbatim from cell 20 of [v1_4_1].
"""

import numpy as np
from itertools import combinations

# protocol constants (decisions D-1 to D-6 above), don't change them
K_SUMMARY      = 3                       # summary_size (locked)
POP_SIZE       = 30                      # MO-WOA locked
MAX_ITER       = 50                      # MO-WOA locked
FE_BUDGET      = 1480                    # 30 + 50*29   (D-3)
KNEE_WEIGHTS   = np.array([32.0, 1.0])   # w_cov = 32 (locked)
REDUNDANCY_OFFSET = 0.2                  # delta in f2
REF_POINT      = np.array([1.1, 1.1])    # HV reference, normalized space


# per-article artefacts, computed once (BGE-M3 is slow)
class ArticleCache:
    """Everything the optimizer needs for one article. No embeddings at run time."""

    def __init__(self, article_id, adj_matrix, node_importance, position_bonus):
        self.article_id        = article_id
        self.adj               = np.asarray(adj_matrix, dtype=float)
        self.node_importance   = np.asarray(node_importance, dtype=float)
        self.position_bonus    = np.asarray(position_bonus, dtype=float)
        self.n                 = len(self.node_importance)
        # same as cell 20: graph.nodes[i]['importance'] * position_bonus[i]
        self.biased_importance = self.node_importance * self.position_bonus

    def to_dict(self):
        return dict(article_id=self.article_id, adj=self.adj,
                    node_importance=self.node_importance,
                    position_bonus=self.position_bonus)

    @staticmethod
    def from_dict(d):
        return ArticleCache(d['article_id'], d['adj'],
                            d['node_importance'], d['position_bonus'])


def build_position_bonus(n):
    """Copied from cell 28 of [v1_4_1], don't touch."""
    pb = np.ones(n)
    pb[0]   = 3.0
    pb[1:3] *= 2.5
    pb[3:5] *= 1.5
    pb[5:8] *= 0.8
    pb[8:]  *= 0.3
    return pb


# objectives, one implementation for all four algorithms
def evaluate_objectives_reference(solution, cache):
    """
    Literal copy of cell 20's evaluate_objectives(), only kept as ground truth for
    G4 (slow double loop). Returns native (coverage max, redundancy min).
    """
    solution = np.asarray(solution).astype(int)
    selected = np.where(solution == 1)[0]
    coverage = cache.biased_importance[selected].sum() if len(selected) > 0 else 0
    redundancy_raw = 0.0
    num_pairs = 0
    for i_idx, i in enumerate(selected):
        for j_idx, j in enumerate(selected):
            if j_idx > i_idx:
                redundancy_raw += cache.adj[i][j]
                num_pairs += 1
    redundancy = redundancy_raw / max(1, num_pairs) if num_pairs > 0 else 0
    redundancy = max(0, redundancy - REDUNDANCY_OFFSET)
    return coverage, redundancy


def evaluate_objectives_vec(combos, cache):
    """Vectorized: (M, k) sorted combos -> (M, 2) native (coverage, redundancy). Checked by G4."""
    combos = np.asarray(combos, dtype=int)
    M, k = combos.shape
    coverage = cache.biased_importance[combos].sum(axis=1)

    n_pairs = k * (k - 1) // 2
    if n_pairs == 0:
        redundancy = np.zeros(M)
    else:
        iu, ju = np.triu_indices(k, k=1)          # all i<j pairs within a subset
        rows = combos[:, iu]                      # (M, n_pairs)
        cols = combos[:, ju]
        redundancy_raw = cache.adj[rows, cols].sum(axis=1)
        redundancy = redundancy_raw / n_pairs
        redundancy = np.maximum(0.0, redundancy - REDUNDANCY_OFFSET)

    return np.column_stack([coverage, redundancy])


# repair operator, copied from cell 20. don't touch
def repair_solution(solution, cache, target_length=K_SUMMARY):
    solution = np.asarray(solution).astype(int).copy()
    bi = cache.biased_importance
    selected = np.where(solution == 1)[0]
    num_selected = len(selected)

    if num_selected > target_length:
        drop_order = selected[np.argsort(bi[selected])]        # lowest importance first
        for idx in drop_order[:num_selected - target_length]:
            solution[idx] = 0
    elif num_selected < target_length:
        unselected = np.where(solution == 0)[0]
        if len(unselected) > 0:
            add_order = unselected[np.argsort(-bi[unselected])]  # highest importance first
            for idx in add_order[:target_length - num_selected]:
                solution[idx] = 1

    if not solution.any():
        solution[np.argmax(bi)] = 1
    return solution


class Enumeration:
    """All C(N, k) subsets of one article: the exact true front (not a pooled one) + O(1) lookups."""

    def __init__(self, cache, k=K_SUMMARY):
        self.cache = cache
        self.k = k
        n = cache.n
        if n < k:
            raise ValueError(f"article {cache.article_id}: N={n} < k={k}")

        self.combos = np.array(list(combinations(range(n), k)), dtype=np.int16)
        self.M = len(self.combos)
        self.F_native = evaluate_objectives_vec(self.combos, cache)   # (M,2) (cov, red)

        # minimization form used by pymoo and every indicator: (-coverage, redundancy)
        self.F_min = np.column_stack([-self.F_native[:, 0], self.F_native[:, 1]])

        # O(1) lookup:  sorted index tuple -> row in F_native
        self.index_of = {tuple(c): i for i, c in enumerate(map(tuple, self.combos))}

        # normalize on the full feasible set, never on algorithm output
        cov = self.F_native[:, 0]
        red = self.F_native[:, 1]
        self.cov_min, self.cov_max = cov.min(), cov.max()
        self.red_min, self.red_max = red.min(), red.max()
        self.cov_range = self.cov_max - self.cov_min
        self.red_range = self.red_max - self.red_min

        # exact true front, deduped in objective space
        nd = non_dominated_mask(self.F_min)
        F_pf = self.F_min[nd]
        self.P_star_min = unique_rows(F_pf)                 # (m, 2) minimization
        self.P_star_norm = self.normalize(self.P_star_min)
        self.m = len(self.P_star_min)                       # |P*|  -> stratum

        # HV of the true front = max attainable HV
        self.hv_star = hypervolume(self.P_star_norm, REF_POINT)

        # true coverage extreme + true knee, for the hit-rate metrics
        self.true_cov_extreme_idx = int(np.argmax(cov))
        self.true_knee_row = self._true_knee()

    def normalize(self, F_min):
        """Map minimization objectives into per-article [0,1]^2 (both minimized)."""
        F_min = np.atleast_2d(np.asarray(F_min, dtype=float))
        cov = -F_min[:, 0]
        red = F_min[:, 1]
        cr = self.cov_range if self.cov_range > 0 else 1.0
        rr = self.red_range if self.red_range > 0 else 1.0
        g1 = (self.cov_max - cov) / cr          # 0 = best coverage
        g2 = (red - self.red_min) / rr          # 0 = least redundant
        return np.column_stack([g1, g2])

    def F_of(self, solution):
        """O(1) lookup for a binary vector, native (coverage, redundancy)."""
        key = tuple(np.where(np.asarray(solution).astype(int) == 1)[0].tolist())
        return self.F_native[self.index_of[key]]

    def row_of(self, solution):
        key = tuple(np.where(np.asarray(solution).astype(int) == 1)[0].tolist())
        return self.index_of[key]

    def _true_knee(self):
        """Knee of the true front with the locked knee_weights = [32, 1]."""
        P = self.P_star_min
        if len(P) == 1:
            return 0
        native = np.column_stack([-P[:, 0], P[:, 1]])
        omin, omax = native.min(axis=0), native.max(axis=0)
        rng = omax - omin
        rng[rng == 0] = 1
        norm = (native - omin) / rng
        norm[:, 0] = 1 - norm[:, 0]                       # invert coverage
        d = np.sqrt((KNEE_WEIGHTS * norm ** 2).sum(axis=1))
        return int(d.argmin())

    @property
    def stratum(self):
        if self.m >= 3:
            return 'A'
        elif self.m == 2:
            return 'B'
        return 'C'


# dominance utils, minimization everywhere
def non_dominated_mask(F):
    """Mask of non-dominated rows, F is (n, 2) minimization."""
    F = np.atleast_2d(np.asarray(F, dtype=float))
    n = len(F)
    keep = np.ones(n, dtype=bool)
    for i in range(n):
        if not keep[i]:
            continue
        # j dominates i  iff  all(F[j] <= F[i]) and any(F[j] < F[i])
        le = np.all(F <= F[i], axis=1)
        lt = np.any(F < F[i], axis=1)
        if np.any(le & lt):
            keep[i] = False
    return keep


def unique_rows(F, decimals=12):
    F = np.atleast_2d(np.asarray(F, dtype=float))
    if len(F) == 0:
        return F
    _, idx = np.unique(np.round(F, decimals), axis=0, return_index=True)
    return F[np.sort(idx)]


def extract_front(F):
    """
    Applied the same way to all four. MOEA/D returns its whole population incl.
    dominated members (30 vs NSGA-II's 6), without this it gets scored on junk.
    """
    F = np.atleast_2d(np.asarray(F, dtype=float))
    if len(F) == 0:
        return F
    return unique_rows(F[non_dominated_mask(F)])


# indicators
def hypervolume(F_norm, ref=REF_POINT):
    """Exact 2-D hypervolume by sweep. F_norm: (n,2) minimization, in [0,1]^2."""
    F = np.atleast_2d(np.asarray(F_norm, dtype=float))
    F = F[np.all(F <= ref, axis=1)]           # drop points outside the ref cuboid
    if len(F) == 0:
        return 0.0
    F = unique_rows(F[non_dominated_mask(F)])
    F = F[np.argsort(F[:, 0])]                # ascending f1
    hv, prev_f2 = 0.0, ref[1]
    for f1, f2 in F:
        if f2 < prev_f2:
            hv += (ref[0] - f1) * (prev_f2 - f2)
            prev_f2 = f2
    return float(hv)


def igd(S_norm, P_norm):
    """IGD: mean distance from each true front point to S."""
    S = np.atleast_2d(S_norm); P = np.atleast_2d(P_norm)
    if len(S) == 0:
        return float('inf')
    d = np.linalg.norm(P[:, None, :] - S[None, :, :], axis=2)
    return float(d.min(axis=1).mean())


def igd_plus(S_norm, P_norm):
    """IGD+ (Ishibuchi et al., 2015) -- weakly Pareto-compliant, unlike IGD."""
    S = np.atleast_2d(S_norm); P = np.atleast_2d(P_norm)
    if len(S) == 0:
        return float('inf')
    diff = np.maximum(S[None, :, :] - P[:, None, :], 0.0)   # (|P|, |S|, 2)
    d = np.sqrt((diff ** 2).sum(axis=2))
    return float(d.min(axis=1).mean())


def gd(S_norm, P_norm):
    """GD: mean distance from each point of S to the true front."""
    S = np.atleast_2d(S_norm); P = np.atleast_2d(P_norm)
    if len(S) == 0:
        return float('inf')
    d = np.linalg.norm(S[:, None, :] - P[None, :, :], axis=2)
    return float(d.min(axis=1).mean())


def spacing(S_norm):
    """Schott (1995). Undefined for |S| < 2 -> NaN (frequency of this is reported)."""
    S = np.atleast_2d(S_norm)
    n = len(S)
    if n < 2:
        return float('nan')
    d = np.abs(S[:, None, :] - S[None, :, :]).sum(axis=2)   # L1
    np.fill_diagonal(d, np.inf)
    di = d.min(axis=1)
    return float(np.sqrt(((di.mean() - di) ** 2).sum() / (n - 1)))


def set_coverage(A, B):
    """C(A,B) = fraction of B weakly dominated by A (Zitzler & Thiele 1998). Not symmetric, report both ways."""
    A = np.atleast_2d(A); B = np.atleast_2d(B)
    if len(B) == 0:
        return float('nan')
    if len(A) == 0:
        return 0.0
    covered = 0
    for b in B:
        le = np.all(A <= b, axis=1)
        lt = np.any(A < b, axis=1)
        if np.any(le & lt) or np.any(np.all(A == b, axis=1)):
            covered += 1
    return covered / len(B)


# FE counter, this is what enforces the budget (D-3)
class EvalLog:
    """
    Logs every new candidate evaluated, in order. 1 FE = 1 new candidate.
    Re-evaluating an unchanged individual doesn't count: pymoo never does it, and
    cell 20 recomputes ~3,100 times per run just for logging.
    """

    def __init__(self, enum, budget=FE_BUDGET):
        self.enum = enum
        self.budget = budget
        self.rows = []        # index into enum.F_native, one per new candidate

    def log(self, solution):
        self.rows.append(self.enum.row_of(solution))
        return self.enum.F_native[self.rows[-1]]

    def log_many(self, X):
        for x in X:
            self.log(x)

    @property
    def n_eval(self):
        return len(self.rows)

    def cumulative_front_min(self):
        """Non-dominated subset of the first `budget` evals, so parity is exact."""
        rows = self.rows[:self.budget]
        if not rows:
            return np.empty((0, 2))
        return extract_front(self.enum.F_min[np.array(rows)])


# shared init (D-1), same P0 for all four
def make_initial_population(cache, seed, pop_size=POP_SIZE, target_length=K_SUMMARY):
    """
    cell 20's initialize_population(), but seeded. All four get this P0 for a given
    (article, seed): no init head start, and the paired design makes Wilcoxon signed-rank legit.
    """
    rng = np.random.default_rng(seed)
    n = cache.n
    init_scores = cache.node_importance * cache.position_bonus
    s = init_scores.sum()
    probs = init_scores / s if s > 0 else np.ones(n) / n

    pop = []
    for _ in range(pop_size):
        sol = np.zeros(n, dtype=int)
        num_to_select = max(1, rng.integers(max(1, target_length - 1), target_length + 2))
        chosen = rng.choice(n, size=min(int(num_to_select), n), replace=False, p=probs)
        sol[chosen] = 1
        pop.append(repair_solution(sol, cache, target_length))
    return np.array(pop, dtype=int)


# MO-WOA, cell 20 logic rewired to the shared layer
def _crowding_distance(objectives):
    """Copied from cell 20's crowding_distance()."""
    objectives = np.atleast_2d(objectives)
    n_sol, n_obj = objectives.shape
    if n_sol <= 2:
        return np.full(n_sol, np.inf)
    dist = np.zeros(n_sol)
    for m in range(n_obj):
        order = np.argsort(objectives[:, m])
        dist[order[0]] = np.inf
        dist[order[-1]] = np.inf
        rng_m = objectives[order[-1], m] - objectives[order[0], m]
        if rng_m == 0:
            continue
        for i in range(1, n_sol - 1):
            dist[order[i]] += (objectives[order[i + 1], m] -
                               objectives[order[i - 1], m]) / rng_m
    return dist


def _pareto_indices(objectives):
    """cell 20's compute_pareto_front in native form (coverage max, redundancy min)."""
    objectives = np.atleast_2d(objectives)
    idx = []
    for i, o1 in enumerate(objectives):
        dominated = False
        for j, o2 in enumerate(objectives):
            if i != j and (o2[0] >= o1[0] and o2[1] <= o1[1] and
                           (o2[0] > o1[0] or o2[1] < o1[1])):
                dominated = True
                break
        if not dominated:
            idx.append(i)
    return idx


def _sigmoid(x):
    """cell 20's sigmoid incl. the [-500, 500] clip. MOPSO uses it too."""
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))


def run_mowoa(enum, seed, pop_size=POP_SIZE, max_iter=MAX_ITER, budget=FE_BUDGET):
    """
    MO-WOA as in cell 20 of [v1_4_1]: binary WOA update, CD leader on the front,
    70/30 leader following, leader kept (so 29 new per gen), repair after every
    update, knee with [32, 1]. FEs: 30 + 50 x 29 = 1480 exactly.
    """
    cache = enum.cache
    rng = np.random.default_rng(seed)
    n = cache.n
    log = EvalLog(enum, budget)

    population = make_initial_population(cache, seed, pop_size)
    log.log_many(population)                                   # 30 FEs

    gen_history = []          # per-generation population objectives (minimization)

    def objs(pop):            # lookups, not counted as FEs (see EvalLog)
        return np.array([enum.F_of(s) for s in pop])

    def update_position_woa(current, guide, a, pop):
        new_pos = np.zeros(n, dtype=int)
        p = rng.random()
        b = 1.0
        A_coeff = 2 * a * rng.random() - a
        C_coeff = 2 * rng.random()

        if p < 0.5:
            if abs(A_coeff) < 1:                                # shrinking encircling
                ref = guide
            else:                                               # random search
                ref = pop[rng.integers(len(pop))]
            for i in range(n):
                D = abs(C_coeff * ref[i] - current[i])
                val = ref[i] - A_coeff * D
                new_pos[i] = 1 if rng.random() < _sigmoid(val) else 0
        else:                                                   # spiral bubble-net
            l = rng.uniform(-1, 1)
            for i in range(n):
                Dp = abs(guide[i] - current[i])
                val = Dp * np.exp(b * l) * np.cos(2 * np.pi * l) + guide[i]
                new_pos[i] = 1 if rng.random() < _sigmoid(val) else 0

        if not new_pos.any():
            new_pos[rng.integers(n)] = 1
        return repair_solution(new_pos, cache)

    for it in range(max_iter):
        a = 2.0 - it * (2.0 / max_iter)
        O = objs(population)
        gen_history.append(np.column_stack([-O[:, 0], O[:, 1]]))

        pidx = _pareto_indices(O)
        if len(pidx) == 0:
            pidx = list(range(len(population)))

        if len(pidx) > 1:
            cd = _crowding_distance(O[pidx])
            inf_mask = np.isinf(cd)
            if np.any(inf_mask):
                lead_in_pareto = rng.choice(np.where(inf_mask)[0])
            else:
                lead_in_pareto = int(np.argmax(cd))
            leader_idx = pidx[lead_in_pareto]
        else:
            leader_idx = pidx[0]
        leader = population[leader_idx]

        new_pop = []
        for i, sol in enumerate(population):
            if i == leader_idx:
                new_pop.append(sol)                             # leader kept
            else:
                guide = leader if rng.random() < 0.7 else population[rng.choice(pidx)]
                child = update_position_woa(sol, guide, a, population)
                log.log(child)                                  # 29 FEs per generation
                new_pop.append(child)
        population = np.array(new_pop)

    O = objs(population)
    gen_history.append(np.column_stack([-O[:, 0], O[:, 1]]))
    F_min_terminal = np.column_stack([-O[:, 0], O[:, 1]])

    # knee point, same as cell 20
    pidx = _pareto_indices(O)
    knee_row = None
    if len(pidx) > 0:
        P = O[pidx]
        omin, omax = P.min(axis=0), P.max(axis=0)
        rngv = omax - omin
        rngv[rngv == 0] = 1
        norm = (P - omin) / rngv
        norm[:, 0] = 1 - norm[:, 0]
        d = np.sqrt((KNEE_WEIGHTS * norm ** 2).sum(axis=1))
        knee_row = enum.row_of(population[pidx[int(d.argmin())]])

    return dict(
        name='MO-WOA',
        terminal_front=extract_front(F_min_terminal),
        cumulative_front=log.cumulative_front_min(),
        gen_history=gen_history,
        n_eval=log.n_eval,
        knee_row=knee_row,
        eval_rows=log.rows,
    )


# binary MOPSO-CD, hand-coded (pymoo's MOPSO is real-valued only)
def run_mopso(enum, seed, pop_size=POP_SIZE, budget=FE_BUDGET,
              w=0.729, c1=1.49445, c2=1.49445, v_max=6.0, archive_size=POP_SIZE):
    """
    Binary MOPSO-CD (Raquel & Naval, 2005). Used instead of the grid MOPSO
    (Coello Coello, Pulido & Lechuga, 2004) because it picks leaders by crowding
    distance like MO-WOA, so the update equation is the only thing that differs:
        particle : w*v + c1*r1*(pbest - x) + c2*r2*(gbest - x)
        whale    : shrinking encircling + spiral bubble-net + random search
    Same sigmoid, repair, init and budget. Clerc constriction defaults, not tuned (D-4).
    """
    cache = enum.cache
    rng = np.random.default_rng(seed)
    n = cache.n
    log = EvalLog(enum, budget)

    B = make_initial_population(cache, seed, pop_size)          # same P0 as MO-WOA (D-1)
    log.log_many(B)                                            # 30 FEs

    # continuous position seeded from the binary start (sigmoid(+/-2) ~ 0.88 / 0.12)
    X = np.where(B == 1, 2.0, -2.0).astype(float)
    V = np.zeros((pop_size, n))

    F = np.array([enum.F_of(b) for b in B])                    # native (cov, red)
    F_min = np.column_stack([-F[:, 0], F[:, 1]])

    pbest_B, pbest_Fmin = B.copy(), F_min.copy()
    arch_B, arch_Fmin = _archive_update(B, F_min, np.empty((0, n), int),
                                        np.empty((0, 2)), archive_size)
    gen_history = [F_min.copy()]

    while log.n_eval < budget:
        # leader: binary tournament on crowding distance
        cd = _crowding_distance(arch_Fmin) if len(arch_Fmin) > 1 else np.array([np.inf])
        for i in range(pop_size):
            if len(arch_B) == 1:
                g = arch_B[0]
            else:
                a1, a2 = rng.integers(len(arch_B), size=2)
                g = arch_B[a1] if cd[a1] >= cd[a2] else arch_B[a2]

            r1, r2 = rng.random(n), rng.random(n)
            V[i] = (w * V[i]
                    + c1 * r1 * (np.where(pbest_B[i] == 1, 2.0, -2.0) - X[i])
                    + c2 * r2 * (np.where(g == 1, 2.0, -2.0) - X[i]))
            V[i] = np.clip(V[i], -v_max, v_max)
            X[i] = X[i] + V[i]
            b = (rng.random(n) < _sigmoid(X[i])).astype(int)   # same sigmoid as MO-WOA
            B[i] = repair_solution(b, cache)                   # same repair as MO-WOA

        F = np.array([enum.F_of(b) for b in B])
        F_min = np.column_stack([-F[:, 0], F[:, 1]])
        log.log_many(B)                                        # 30 new candidates
        gen_history.append(F_min.copy())

        # pbest by dominance, tie -> keep new with prob 0.5
        for i in range(pop_size):
            new, old = F_min[i], pbest_Fmin[i]
            if np.all(new <= old) and np.any(new < old):
                pbest_B[i], pbest_Fmin[i] = B[i].copy(), new
            elif not (np.all(old <= new) and np.any(old < new)):
                if rng.random() < 0.5:
                    pbest_B[i], pbest_Fmin[i] = B[i].copy(), new

        arch_B, arch_Fmin = _archive_update(B, F_min, arch_B, arch_Fmin, archive_size)

    return dict(
        name='MOPSO',
        terminal_front=extract_front(arch_Fmin),               # native output is the archive
        cumulative_front=log.cumulative_front_min(),
        gen_history=gen_history,
        n_eval=log.n_eval,
        knee_row=None,
        eval_rows=log.rows,
    )


def _archive_update(B, F_min, arch_B, arch_Fmin, cap):
    """Merge population into the external archive; prune by crowding distance."""
    allB = np.vstack([arch_B, B]) if len(arch_B) else B.copy()
    allF = np.vstack([arch_Fmin, F_min]) if len(arch_Fmin) else F_min.copy()

    keep = non_dominated_mask(allF)
    allB, allF = allB[keep], allF[keep]

    _, uidx = np.unique(np.round(allF, 12), axis=0, return_index=True)
    allB, allF = allB[np.sort(uidx)], allF[np.sort(uidx)]

    if len(allF) > cap:                                        # prune the most crowded
        cd = _crowding_distance(allF)
        keep_idx = np.argsort(-cd)[:cap]
        allB, allF = allB[keep_idx], allF[keep_idx]
    return allB, allF


# pymoo adapters for NSGA-II and MOEA/D
def _pymoo_imports():
    from pymoo.core.problem import Problem
    from pymoo.core.repair import Repair
    from pymoo.core.sampling import Sampling
    from pymoo.core.callback import Callback
    return Problem, Repair, Sampling, Callback


def make_pymoo_components(enum, seed, budget=FE_BUDGET, pop_size=POP_SIZE):
    Problem, Repair, Sampling, Callback = _pymoo_imports()
    cache = enum.cache
    log = EvalLog(enum, budget)

    class SummaryProblem(Problem):
        def __init__(self):
            # needs vtype=bool
            super().__init__(n_var=cache.n, n_obj=2, xl=0, xu=1, vtype=bool)

        def _evaluate(self, X, out, **kw):
            Xi = np.asarray(X).astype(int)
            F = np.array([enum.F_of(x) for x in Xi])
            log.log_many(Xi)                                   # every row = 1 new FE
            out["F"] = np.column_stack([-F[:, 0], F[:, 1]])    # minimization form

    class SummaryRepair(Repair):
        def _do(self, problem, X, **kw):
            # has to return bool. with int64, BitflipMutation flips via bitwise NOT
            # (~0 = -1, ~1 = -2), the encoding gets corrupted and the run still
            # finishes without error (cardinality sums go negative). hit this in testing
            return np.array([repair_solution(x, cache) for x in np.asarray(X).astype(int)]).astype(bool)

    class SharedSampling(Sampling):
        """D-1: every algorithm gets MO-WOA's importance-biased P0."""
        def _do(self, problem, n_samples, **kw):
            P0 = make_initial_population(cache, seed, pop_size)
            if n_samples <= len(P0):
                return P0[:n_samples].astype(bool)
            extra = make_initial_population(cache, seed + 100000,
                                            n_samples - len(P0))
            return np.vstack([P0, extra]).astype(bool)

    class GenCallback(Callback):
        def __init__(self):
            super().__init__()
            self.gen_history = []
        def notify(self, algorithm):
            self.gen_history.append(algorithm.pop.get("F").copy())

    return SummaryProblem(), SummaryRepair(), SharedSampling(), GenCallback(), log


def run_pymoo(enum, seed, which, budget=FE_BUDGET, pop_size=POP_SIZE):
    """
    which: 'NSGA-II' | 'MOEA/D'
    pymoo only checks termination at generation boundaries, so it can't stop at
    exactly 1480 (NSGA-II with ('n_evals', 1500) used 1524). Peers overshoot to
    the next boundary, so the surplus goes to them, never MO-WOA. The cumulative
    front is cut at the first 1480 evals, so parity is exact there.
    """
    from pymoo.algorithms.moo.nsga2 import NSGA2
    from pymoo.algorithms.moo.moead import MOEAD
    from pymoo.operators.crossover.pntx import TwoPointCrossover
    from pymoo.operators.mutation.bitflip import BitflipMutation
    from pymoo.util.ref_dirs import get_reference_directions
    from pymoo.termination.max_eval import MaximumFunctionCallTermination
    from pymoo.optimize import minimize

    problem, repair, sampling, cb, log = make_pymoo_components(enum, seed, budget, pop_size)

    common = dict(sampling=sampling, crossover=TwoPointCrossover(),
                  mutation=BitflipMutation(), repair=repair)

    if which == 'NSGA-II':
        algo = NSGA2(pop_size=pop_size, eliminate_duplicates=True, **common)
    elif which == 'MOEA/D':
        ref_dirs = get_reference_directions("uniform", 2, n_partitions=pop_size - 1)
        algo = MOEAD(ref_dirs=ref_dirs, n_neighbors=15, **common)
    else:
        raise ValueError(which)

    res = minimize(problem, algo, MaximumFunctionCallTermination(budget),
                   seed=seed, callback=cb, verbose=False, save_history=False)

    # MOEA/D returns its whole population, extract_front cleans that up (same for all four)
    F_terminal = np.atleast_2d(res.pop.get("F"))

    return dict(
        name=which,
        terminal_front=extract_front(F_terminal),
        cumulative_front=log.cumulative_front_min(),
        gen_history=cb.gen_history,
        n_eval=log.n_eval,
        knee_row=None,
        eval_rows=log.rows,
    )


# one (article, seed) -> all four algorithms
ALGORITHMS = ['MO-WOA', 'NSGA-II', 'MOPSO', 'MOEA/D']


def run_all(enum, seed):
    return {
        'MO-WOA':  run_mowoa(enum, seed),
        'NSGA-II': run_pymoo(enum, seed, 'NSGA-II'),
        'MOPSO':   run_mopso(enum, seed),
        'MOEA/D':  run_pymoo(enum, seed, 'MOEA/D'),
    }


def score_run(result, enum, tier='terminal'):
    """Compute every indicator for one run. tier: 'terminal' | 'cumulative' (D-2)."""
    F = result[f'{tier}_front']
    S = enum.normalize(F) if len(F) else np.empty((0, 2))
    P = enum.P_star_norm

    hv = hypervolume(S)
    out = {
        'algorithm': result['name'],
        'tier': tier,
        'HV': hv,
        'HV_pct': 100.0 * hv / enum.hv_star if enum.hv_star > 0 else float('nan'),
        'IGD': igd(S, P),
        'IGD+': igd_plus(S, P),
        'GD': gd(S, P),
        'Spacing': spacing(S),
        'cardinality': len(S),
        'n_eval': result['n_eval'],
    }
    # only possible because we know the true front
    if len(F):
        pf = unique_rows(enum.P_star_min)
        hits = sum(1 for f in F if np.any(np.all(np.isclose(pf, f), axis=1)))
        out['front_recovery'] = hits / len(pf)
        cov_ext = enum.F_min[enum.true_cov_extreme_idx]
        out['found_cov_extreme'] = bool(np.any(np.all(np.isclose(F, cov_ext), axis=1)))
    else:
        out['front_recovery'] = 0.0
        out['found_cov_extreme'] = False
    return out


def hv_convergence(result, enum):
    """HV-vs-generation, normalized as a % of the true optimal HV."""
    return [100.0 * hypervolume(enum.normalize(extract_front(Fg))) / enum.hv_star
            if enum.hv_star > 0 else float('nan')
            for Fg in result['gen_history']]


# MO-WOA+, the fixed variant. canonical MO-WOA above stays untouched.
# pilot diagnosis (real data):
#   retention: terminal 75.70% vs cumulative 87.28% HV, 11.58 pts thrown away.
#     no archive, generational replacement drops what it found
#   search: S-transfer saturates. as a->0, A->0, val -> leader[i] in {0,1},
#     sigmoid(val) -> {0.50, 0.73}, so every bit is a near coin-flip and
#     repair_solution() collapses the offspring onto greedy top-3. diversity dies,
#     search only adds ~4.6 pts in 1480 FEs
# fixes, each switchable for the ablation:
#   use_archive: external archive, cap 30 like MOPSO, non-dominated, crowding-pruned
#     [Coello Coello et al., 2004]
#   transfer='V': V-shaped transfer [Mirjalili & Lewis, 2013; binary WOA: Hussien
#     et al., 2019]. step magnitude -> flip prob of the parent's bit, step->0 keeps the parent

def _v_transfer(step):
    """V-shaped transfer: |tanh(step)|. V(0)=0 (keep parent bit), V(large)->1."""
    return np.abs(np.tanh(step))


def _nds_crowding_select(B_all, F_all, k):
    """(mu+lambda) environmental selection: non-dominated sorting + crowding (NSGA-II)."""
    Nn = len(F_all)
    remaining = np.arange(Nn)
    chosen = []
    while len(chosen) < k and len(remaining) > 0:
        mask = non_dominated_mask(F_all[remaining])
        front = remaining[mask]
        if len(chosen) + len(front) <= k:
            chosen.extend(front.tolist())
            remaining = remaining[~mask]
        else:
            cd = _crowding_distance(F_all[front])
            need = k - len(chosen)
            chosen.extend(front[np.argsort(-cd)[:need]].tolist())
            break
    chosen = np.array(chosen[:k])
    return B_all[chosen], F_all[chosen]


def run_mowoa_final(enum, seed, transfer='V', guide_mode='per_whale_archive',
                    survival='elitist', use_archive=True,
                    ls_probes=0, ls_mode='directed',
                    pop_size=POP_SIZE, max_iter=MAX_ITER, budget=FE_BUDGET,
                    archive_size=POP_SIZE):
    """
    Ablation-switchable MO-WOA+. Update equations, repair, init (D-1), budget (D-3)
    and knee weights are the same as cell 20. Switches:
      transfer   : 'S' (cell 20 sigmoid) | 'V' (V-shaped flip)
      guide_mode : 'single_leader' (cell 20) | 'per_whale_archive' (MOPSO-CD gbest rule)
      survival   : 'generational' (cell 20) | 'elitist' (mu+lambda NDS+crowding, like NSWOA)
      use_archive: external archive, cap 30, crowding-pruned
    S / single_leader / generational / no archive reproduces run_mowoa() bit-exactly (G7b).
    """
    cache = enum.cache
    rng = np.random.default_rng(seed)
    n = cache.n
    log = EvalLog(enum, budget)

    population = make_initial_population(cache, seed, pop_size)
    log.log_many(population)

    def objs(pop):
        return np.array([enum.F_of(s) for s in pop])

    F = objs(population)
    F_min = np.column_stack([-F[:, 0], F[:, 1]])
    gen_history = [F_min.copy()]

    arch_B = np.empty((0, n), dtype=int); arch_F = np.empty((0, 2))
    if use_archive:
        arch_B, arch_F = _archive_update(population, F_min, arch_B, arch_F, archive_size)

    def offspring_of(current, guide, a, pop):
        p = rng.random(); b = 1.0
        A_coeff = 2 * a * rng.random() - a
        C_coeff = 2 * rng.random()
        if transfer == 'S':
            new_pos = np.zeros(n, dtype=int)
            if p < 0.5:
                ref = guide if abs(A_coeff) < 1 else pop[rng.integers(len(pop))]
                for i in range(n):
                    D = abs(C_coeff * ref[i] - current[i])
                    val = ref[i] - A_coeff * D
                    new_pos[i] = 1 if rng.random() < _sigmoid(val) else 0
            else:
                l = rng.uniform(-1, 1)
                for i in range(n):
                    Dp = abs(guide[i] - current[i])
                    val = Dp * np.exp(b * l) * np.cos(2 * np.pi * l) + guide[i]
                    new_pos[i] = 1 if rng.random() < _sigmoid(val) else 0
        else:
            new_pos = current.copy()
            if p < 0.5:
                ref = guide if abs(A_coeff) < 1 else pop[rng.integers(len(pop))]
                for i in range(n):
                    D = abs(C_coeff * ref[i] - current[i])
                    if rng.random() < _v_transfer(A_coeff * D):
                        new_pos[i] = 1 - new_pos[i]
            else:
                l = rng.uniform(-1, 1)
                for i in range(n):
                    Dp = abs(guide[i] - current[i])
                    if rng.random() < _v_transfer(Dp * np.exp(b * l) * np.cos(2 * np.pi * l)):
                        new_pos[i] = 1 - new_pos[i]
        if not new_pos.any():
            new_pos[rng.integers(n)] = 1
        return repair_solution(new_pos, cache)

    def ls_probe():
        """1-swap probe on an archive member. Cardinality stays at k so repair never
        fires, it's the only move that skips repair's importance bias.
        random: uniform swap. directed: drop the selected sentence most similar to
        the others, add one of the 3 least similar unselected ones."""
        if len(arch_B) == 0:
            return None
        if len(arch_B) == 1:
            base = arch_B[0]
        else:
            cdp = _crowding_distance(arch_F)
            a1, a2 = rng.integers(len(arch_B), size=2)
            base = arch_B[a1] if cdp[a1] >= cdp[a2] else arch_B[a2]
        x = np.asarray(base).astype(int).copy()
        sel = np.where(x == 1)[0]
        unsel = np.where(x == 0)[0]
        if len(sel) == 0 or len(unsel) == 0:
            return None
        if ls_mode == 'random':
            out_i = rng.choice(sel)
            in_i = rng.choice(unsel)
        else:
            contrib = np.array([cache.adj[i, sel].sum() for i in sel])
            out_i = sel[int(np.argmax(contrib))] if rng.random() < 0.7 else rng.choice(sel)
            remaining = sel[sel != out_i]
            scores = cache.adj[unsel][:, remaining].sum(axis=1)
            k = min(3, len(unsel))
            in_i = unsel[rng.choice(np.argsort(scores)[:k])]
        x[out_i] = 0
        x[in_i] = 1
        return x

    for it in range(max_iter):
        a = 2.0 - it * (2.0 / max_iter)
        F = objs(population)
        F_min = np.column_stack([-F[:, 0], F[:, 1]])
        pidx = _pareto_indices(F)
        if len(pidx) == 0:
            pidx = list(range(len(population)))

        # leader / guides
        if guide_mode == 'single_leader':
            if len(pidx) > 1:
                cd = _crowding_distance(F[pidx])
                inf_mask = np.isinf(cd)
                lp = (rng.choice(np.where(inf_mask)[0]) if np.any(inf_mask)
                      else int(np.argmax(cd)))
                leader_idx = pidx[lp]
            else:
                leader_idx = pidx[0]
            leader = population[leader_idx]
        else:
            leader_idx = 0
            leader = None

        def draw_guide():
            if guide_mode == 'per_whale_archive' and use_archive and len(arch_B) > 0:
                if len(arch_B) == 1:
                    return arch_B[0]
                cd = _crowding_distance(arch_F)
                a1, a2 = rng.integers(len(arch_B), size=2)
                return arch_B[a1] if cd[a1] >= cd[a2] else arch_B[a2]
            if guide_mode == 'per_whale_archive':          # archive off: pop front
                if len(pidx) == 1:
                    return population[pidx[0]]
                cd = _crowding_distance(F[pidx])
                a1, a2 = rng.integers(len(pidx), size=2)
                return population[pidx[a1] if cd[a1] >= cd[a2] else pidx[a2]]
            return leader

        # 29 offspring, leader stays in its slot like cell 20 (needed for the
        # bit-exact rng stream at baseline switches)
        new_pop = []
        children = []
        if ls_probes > 0 and use_archive and len(arch_B) > 0:
            nonleader = [i for i in range(len(population)) if i != leader_idx]
            probe_slots = set(rng.choice(nonleader,
                                         size=min(ls_probes, len(nonleader)),
                                         replace=False).tolist())
        else:
            probe_slots = set()
        for i, sol in enumerate(population):
            if i == leader_idx:
                new_pop.append(sol)
                continue
            child = None
            if i in probe_slots:
                child = ls_probe()
            if child is None:
                g = draw_guide() if rng.random() < 0.7 else population[rng.choice(pidx)]
                child = offspring_of(sol, g, a, population)
            log.log(child)
            new_pop.append(child)
            children.append(child)
        children = np.array(children)
        Fc = objs(children)
        Fc_min = np.column_stack([-Fc[:, 0], Fc[:, 1]])

        # survival
        if survival == 'generational':
            population = np.array(new_pop)
        else:                                              # elitist (mu+lambda)
            B_all = np.vstack([population, children])
            F_all = np.vstack([F_min, Fc_min])
            population, _ = _nds_crowding_select(B_all, F_all, pop_size)

        Fq = objs(population)
        Fq_min = np.column_stack([-Fq[:, 0], Fq[:, 1]])
        gen_history.append(Fq_min.copy())
        if use_archive:
            arch_B, arch_F = _archive_update(children, Fc_min, arch_B, arch_F,
                                             archive_size)

    # terminal output + knee
    if use_archive:
        F_terminal = arch_F; knee_B, knee_F = arch_B, arch_F
    else:
        Fq = objs(population)
        F_terminal = np.column_stack([-Fq[:, 0], Fq[:, 1]])
        pidx = _pareto_indices(Fq)
        knee_B, knee_F = population[pidx], F_terminal[pidx]

    knee_row = None
    if len(knee_F) > 0:
        native = np.column_stack([-knee_F[:, 0], knee_F[:, 1]])
        omin, omax = native.min(axis=0), native.max(axis=0)
        rngv = omax - omin; rngv[rngv == 0] = 1
        norm = (native - omin) / rngv
        norm[:, 0] = 1 - norm[:, 0]
        d = np.sqrt((KNEE_WEIGHTS * norm ** 2).sum(axis=1))
        knee_row = enum.row_of(knee_B[int(d.argmin())])

    return dict(name=f"MO-WOA+[{transfer},{guide_mode[:8]},{survival[:4]}"
                     f"{',arch' if use_archive else ''}]",
                terminal_front=extract_front(F_terminal),
                cumulative_front=log.cumulative_front_min(),
                gen_history=gen_history, n_eval=log.n_eval,
                knee_row=knee_row, eval_rows=log.rows)

## D3 - load caches, enumeration and phase C checkpoints (from phase D)

In [ ]:
import importlib, mowoa_mo_shared as mo
importlib.reload(mo)

with open(CACHE_PATH, 'rb') as f:
    caches = [mo.ArticleCache.from_dict(d) for d in pickle.load(f)]
t0 = time.time()
enums = {c.article_id: mo.Enumeration(c) for c in caches}
print(f'enumeration rebuilt in {time.time()-t0:.1f}s')

records = []
fronts  = {}          # (article, algorithm, seed) -> terminal front (min. form)
curves  = {}          # (article, algorithm, seed) -> HV% per generation
knees   = {}          # (article, algorithm, seed) -> knee row (or None)
n_files = 0
for aid in sorted(enums):
    ck = f'{PHASEC_DIR}/article_{aid:03d}.pkl'
    with open(ck, 'rb') as f:
        rows = pickle.load(f)
    n_files += 1
    for r in rows:
        key = (r['article'], r['algorithm'], r['seed'])
        fronts[key] = np.asarray(r['terminal_front'])
        curves[key] = list(r['hv_curve'])
        knees[key]  = r['knee_row']
        rec = {k: r[k] for k in r
               if k not in ('terminal_front', 'cumulative_front', 'hv_curve')}
        records.append(rec)

df = pd.DataFrame(records)
assert n_files == N_ARTICLES, f'only {n_files}/{N_ARTICLES} checkpoints found'
print(f'{n_files} checkpoints, {len(df)} runs')
print(df.groupby('algorithm').size())

HARD = sorted(df.loc[df.hard, 'article'].unique().tolist())
EASY = sorted(df.loc[~df.hard, 'article'].unique().tolist())
print(f'hard articles: {len(HARD)}   easy: {len(EASY)}')
dh = df[df.hard].copy()

## [W0] configuration

In [ ]:
!pip install kagglehub rouge-score -q
import os, json, re, time, pickle, inspect, platform, collections
import numpy as np, pandas as pd, networkx as nx
from rouge_score import rouge_scorer, tokenize as rs_tokenize
from scipy import stats

assert all(v in globals() for v in ('mo', 'caches', 'enums', 'knees')), \
    'Run D1, D2 and D3 of the Phase D notebook first.'

WP1_DIR  = f'{DRIVE_DIR}/wp1';            os.makedirs(WP1_DIR, exist_ok=True)
MP_DIR   = f'{WP1_DIR}/mowoa_plus_runs';  os.makedirs(MP_DIR, exist_ok=True)
CANONICAL_JSON = f'{DRIVE_DIR}/mowoa_summaries_150_rc1_w32_update_test.json'
INDOBERT_DIR   = '/content/drive/MyDrive/thesis_mowoa_cache'

SEEDS = list(range(30))   # phase C seeds. [0] is a quick pass (~3 min), running
                          # again with range(30) only computes the missing seeds
WEIGHTS = {'w1': [1.0, 1.0], 'w2': [2.0, 1.0], 'w4': [4.0, 1.0], 'w8': [8.0, 1.0],
           'w16': [16.0, 1.0], 'w32': [32.0, 1.0], 'w64': [64.0, 1.0]}
FROZEN = dict(transfer='V', guide_mode='per_whale_archive', survival='elitist',
              use_archive=True, ls_probes=10, ls_mode='directed')     # = Phase C Cell 13
SUMMARY_SIZE = 3
LEAD3 = (0, 1, 2)
sc = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)  # locked

LOG = []
def out(*a):
    s = ' '.join(str(x) for x in a)
    print(s)
    LOG.append(s)

with open(CANONICAL_JSON, encoding='utf-8') as f:
    canon = json.load(f)
golden = canon['golden_summaries'][:N_ARTICLES]
with open(f'{PHASEC_DIR}/sentences.pkl', 'rb') as f:
    SENTS = pickle.load(f)
cache_by_id = {c.article_id: c for c in caches}
AIDS = sorted(enums)
out('=' * 78)
out('WP1 / C1 run —', time.strftime('%Y-%m-%d %H:%M'), '| python', platform.python_version(),
    '| CPU', platform.processor() or platform.machine(), 'x', os.cpu_count())
out('references', len(golden), '| sentence lists', len(SENTS), '| enumerations', len(enums),
    '| seeds', SEEDS[0], '..', SEEDS[-1])

## [W1] dataset checks: Q8, Q23, Q24, fold disjointness, Tabel 4.1

In [ ]:
import kagglehub
base_path = kagglehub.dataset_download("linkgish/indosum")
path = f"{base_path}/indosum"

# copied from [v1_4_4] cell 11 / [eval_v1_4_1] CELL 2
def split_text_into_sentences(paragraphs):
    sentences = []
    for paragraph in paragraphs:
        for sentence_tokens in paragraph:
            sentence = ' '.join(sentence_tokens)
            sentence = sentence.replace(' ,', ',')
            sentence = sentence.replace(' .', '.')
            sentence = sentence.replace(' :', ':')
            sentence = sentence.replace(' ;', ';')
            sentence = sentence.replace(' !', '!')
            sentence = sentence.replace(' ?', '?')
            sentence = sentence.replace(' )', ')')
            sentence = sentence.replace('( ', '(')
            if sentence.strip():
                sentences.append(sentence.strip())
    return sentences

# copied from [v1_4_4] cell 27
def tokens_to_narrative(tokenized_sentences):
    sentences = []
    for tokens in tokenized_sentences:
        sentence = " ".join(tokens)
        sentence = sentence.replace(" ,", ",")
        sentence = sentence.replace(" .", ".")
        sentence = sentence.replace(" :", ":")
        sentence = sentence.replace(" ;", ";")
        sentence = sentence.replace(" !", "!")
        sentence = sentence.replace(" ?", "?")
        sentences.append(sentence)
    return " ".join(sentences)

test_full = pd.read_json(f'{path}/test.01.jsonl', lines=True)
out('\n[W1] DATASET')
out('test.01.jsonl articles:', len(test_full), '| columns:', list(test_full.columns))
ok_sent = sum(split_text_into_sentences(test_full.paragraphs[i]) == SENTS[i] for i in range(N_ARTICLES))
ok_src  = sum(' '.join(split_text_into_sentences(test_full.paragraphs[i])) == canon['source_texts'][i]
              for i in range(N_ARTICLES))
ok_ref  = sum(tokens_to_narrative(test_full.summary[i]) == golden[i] for i in range(N_ARTICLES))
out(f'first 150 of test.01 vs Phase C sentences.pkl: {ok_sent}/150 identical sentence lists')
out(f'first 150 of test.01 vs JSON source_texts   : {ok_src}/150 identical')
out(f'first 150 of test.01 vs JSON golden summaries: {ok_ref}/150 identical')

id_col = 'id' if 'id' in test_full.columns else None
if id_col:
    ids = pd.DataFrame({'article': np.arange(1, N_ARTICLES + 1), 'index_in_test01': np.arange(N_ARTICLES),
                        'indosum_id': test_full[id_col][:N_ARTICLES].values})
    for extra in ('category', 'source'):
        if extra in test_full.columns:
            ids[extra] = test_full[extra][:N_ARTICLES].values
    ids.to_csv(f'{WP1_DIR}/wp1_ids.csv', index=False)
    train_full = pd.read_json(f'{path}/train.01.jsonl', lines=True)
    tr_ids = set(train_full[id_col])
    out(f'IDs saved (wp1_ids.csv). train.01 ∩ test.01 ids: {len(tr_ids & set(test_full[id_col]))}; '
        f'evaluated 150 found in train.01: {sum(i in tr_ids for i in ids.indosum_id)}')
    out('first ids:', list(ids.indosum_id[:3]))
else:
    train_full = pd.read_json(f'{path}/train.01.jsonl', lines=True)
    out('no id column in test.01 — Q23 cannot be answered from the file')

out('files in the download:', sorted(os.listdir(path))[:40])
for d in dict.fromkeys([base_path, path]):
    for fn in sorted(os.listdir(d)):
        fp = f'{d}/{fn}'
        if os.path.isfile(fp) and re.match(r'(readme|changelog|licen)', fn, re.I):
            txt = open(fp, encoding='utf-8', errors='replace').read().splitlines()
            hits = [l for l in txt if re.search(r'licen|copyright|redistribut|fold|cite|split', l, re.I)]
            out(f'--- {fn}: {len(txt)} lines; lines mentioning licence/fold/citation/split:')
            for l in hits[:25]:
                out('   ', l.strip()[:160])

# Tabel 4.1 again, now on the evaluated 150 (feature functions copied from v1.4.4 cell 7)
def n_paragraphs(p):         return len(p)
def n_sentences_article(p):  return sum(len(para) for para in p)
def n_words_article(p):      return sum(len(s) for para in p for s in para)
def avg_words_per_sent(p):
    lens = [len(s) for para in p for s in para]
    return np.mean(lens) if lens else 0.0
def n_sentences_summary(s):  return len(s)
def n_words_summary(s):      return sum(len(sent) for sent in s)
def comp_ratio_sent(p, s):
    n = n_sentences_article(p)
    return n_sentences_summary(s) / n if n > 0 else 0.0
def comp_ratio_word(p, s):
    n = n_words_article(p)
    return n_words_summary(s) / n if n > 0 else 0.0
def build_vocab(df):
    vocab = set()
    for paragraphs in df['paragraphs']:
        for para in paragraphs:
            for sent in para:
                vocab.update(w.lower() for w in sent)
    return vocab

eval_df = test_full.iloc[:N_ARTICLES][['paragraphs', 'summary']].reset_index(drop=True)
feat = pd.DataFrame()
feat['n_paragraphs']        = eval_df['paragraphs'].apply(n_paragraphs)
feat['n_sentences_article'] = eval_df['paragraphs'].apply(n_sentences_article)
feat['n_words_article']     = eval_df['paragraphs'].apply(n_words_article)
feat['avg_words_per_sent']  = eval_df['paragraphs'].apply(avg_words_per_sent)
feat['n_sentences_summary'] = eval_df['summary'].apply(n_sentences_summary)
feat['n_words_summary']     = eval_df['summary'].apply(n_words_summary)
feat['comp_ratio_sent']     = eval_df.apply(lambda r: comp_ratio_sent(r['paragraphs'], r['summary']), axis=1)
feat['comp_ratio_word']     = eval_df.apply(lambda r: comp_ratio_word(r['paragraphs'], r['summary']), axis=1)
t41 = [('Total Dokumen', len(eval_df)),
       ('Ukuran Kosakata (Token Unik)', len(build_vocab(eval_df))),
       ('Rata-rata Paragraf per Artikel', feat.n_paragraphs.mean()),
       ('Rata-rata Kalimat per Artikel', feat.n_sentences_article.mean()),
       ('Minimum Kalimat per Artikel', feat.n_sentences_article.min()),
       ('Maksimum Kalimat per Artikel', feat.n_sentences_article.max()),
       ('Rata-rata Kata per Artikel', feat.n_words_article.mean()),
       ('Rata-rata Kata per Kalimat (Artikel)', feat.avg_words_per_sent.mean()),
       ('Rata-rata Kalimat per Ringkasan', feat.n_sentences_summary.mean()),
       ('Minimum Kalimat per Ringkasan', feat.n_sentences_summary.min()),
       ('Maksimum Kalimat per Ringkasan', feat.n_sentences_summary.max()),
       ('Rata-rata Kata per Ringkasan', feat.n_words_summary.mean()),
       ('Rata-rata Rasio Kompresi (Kalimat)', feat.comp_ratio_sent.mean()),
       ('Rata-rata Rasio Kompresi (Kata)', feat.comp_ratio_word.mean())]
pd.DataFrame(t41, columns=['Metrik', 'Nilai']).to_csv(f'{WP1_DIR}/wp1_tabel41.csv', index=False)
out('\nTABEL 4.1 on the evaluated 150 (first 150 of test.01; IndoSum tokens incl. punctuation)')
for k, v in t41:
    out(f'  {k:<40} {v:.3f}' if isinstance(v, float) else f'  {k:<40} {v}')
out('  reference sentences per summary: distribution',
    dict(sorted(collections.Counter(feat.n_sentences_summary).items())))
# where the current Tabel 4.1 came from: cell 7 = random 150 of test.01-05, random_state=42
import glob
_old = pd.concat([pd.read_json(f, lines=True)[['paragraphs', 'summary']]
                  for f in sorted(glob.glob(f'{path}/test.*.jsonl'))], ignore_index=True)
_old = _old.sample(n=150, random_state=42).reset_index(drop=True)
out(f'  cell-7 sample (test.01-05, seed 42): avg words/summary {_old.summary.apply(n_words_summary).mean():.2f}, '
    f'avg sentences/article {_old.paragraphs.apply(n_sentences_article).mean():.2f} '
    f'(current Tabel 4.1: 68.21, 19.47)')

## [W2] native LEAD-3, TextRank, LexRank (functions copied from eval CELLs 6-8)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity

def lead_n_summarize(sentences, n=3):
    selected = sentences[:n]
    return ' '.join(selected)

def textrank_summarize(sentences, n=3):
    if len(sentences) <= n:
        return ' '.join(sentences)
    try:
        vectorizer = TfidfVectorizer(max_features=5000)
        tfidf_matrix = vectorizer.fit_transform(sentences)
    except ValueError:
        return ' '.join(sentences[:n])
    sim_matrix = sklearn_cosine_similarity(tfidf_matrix)
    G = nx.Graph()
    for i in range(len(sentences)):
        G.add_node(i)
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            if sim_matrix[i, j] > 0:
                G.add_edge(i, j, weight=sim_matrix[i, j])
    if len(G.edges()) == 0:
        scores = {i: 1.0 / len(sentences) for i in range(len(sentences))}
    else:
        scores = nx.pagerank(G, weight='weight', alpha=0.85)
    ranked = sorted(scores.keys(), key=lambda i: scores[i], reverse=True)
    selected_indices = sorted(ranked[:n])
    return ' '.join([sentences[i] for i in selected_indices])

def lexrank_summarize(sentences, n=3, threshold=0.1):
    if len(sentences) <= n:
        return ' '.join(sentences)
    try:
        vectorizer = TfidfVectorizer(max_features=5000)
        tfidf_matrix = vectorizer.fit_transform(sentences)
    except ValueError:
        return ' '.join(sentences[:n])
    sim_matrix = sklearn_cosine_similarity(tfidf_matrix)
    G = nx.Graph()
    for i in range(len(sentences)):
        G.add_node(i)
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            if sim_matrix[i, j] > threshold:
                G.add_edge(i, j, weight=sim_matrix[i, j])
    if len(G.edges()) == 0:
        scores = {i: 1.0 / len(sentences) for i in range(len(sentences))}
    else:
        scores = nx.pagerank(G, weight='weight', alpha=0.85)
    ranked = sorted(scores.keys(), key=lambda i: scores[i], reverse=True)
    selected_indices = sorted(ranked[:n])
    return ' '.join([sentences[i] for i in selected_indices])

SUMM = {}   # algorithm -> list of 150 summary strings, native sentence units
SUMM['LEAD-3']   = [lead_n_summarize(SENTS[a], n=SUMMARY_SIZE) for a in AIDS]
SUMM['TextRank'] = [textrank_summarize(SENTS[a], n=SUMMARY_SIZE) for a in AIDS]
SUMM['LexRank']  = [lexrank_summarize(SENTS[a], n=SUMMARY_SIZE, threshold=0.1) for a in AIDS]
out('\n[W2] LEAD-3 / TextRank / LexRank done on native sentences')

# check: the old regex units of [eval_v1_4_1] CELL 4 (copied as is) should give
# the current Tabel 4.18 rows (R-1 LEAD-3 0.6484, TextRank 0.4229, LexRank 0.4211)
def split_canonical_source(source_text):
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z"\'\(])', source_text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences
REGEX = {a: split_canonical_source(canon['source_texts'][a]) for a in AIDS}
r1 = lambda texts: np.mean([sc.score(golden[a], t)['rouge1'].fmeasure for a, t in zip(AIDS, texts)])
out(f'  regex units: {np.mean([len(REGEX[a]) for a in AIDS]):.2f} sentences/article vs native '
    f'{np.mean([len(SENTS[a]) for a in AIDS]):.2f}; R-1 reproduction (Tabel 4.18 now): '
    f'LEAD-3 {r1([lead_n_summarize(REGEX[a]) for a in AIDS]):.4f} (0.6484), '
    f'TextRank {r1([textrank_summarize(REGEX[a]) for a in AIDS]):.4f} (0.4229), '
    f'LexRank {r1([lexrank_summarize(REGEX[a]) for a in AIDS]):.4f} (0.4211)')

## [W3] single-objective WOA on native sentences, graph from the phase B cache

In [ ]:
# single_objective_woa copied as is from [eval_v1_4_1] CELL 9. the graph is rebuilt
# from the cached BGE-M3 adjacency (edges where cos > 0.1) and LexRank importance,
# so it gets the same inputs as MO-WOA+ (no re-embedding of regex sentences)
def single_objective_woa(graph, pop_size=30, max_iter=50,
                          summary_size=3, position_bonus=None):
    num_sentences = len(graph.nodes())
    if summary_size >= 1:
        target_length = max(1, min(int(summary_size), num_sentences))
    else:
        target_length = max(1, int(num_sentences * summary_size))

    node_importance = np.array([
        graph.nodes[n].get('importance', 0) for n in graph.nodes()
    ])
    if np.all(node_importance == 0):
        node_importance = np.array([
            graph.degree(n, weight='weight') for n in graph.nodes()
        ])
    adj_matrix = nx.to_numpy_array(graph, weight='weight')

    if position_bonus is None:
        position_bonus = np.ones(num_sentences)

    biased_importance = np.array([
        graph.nodes[i]['importance'] * position_bonus[i]
        for i in range(num_sentences)
    ])

    def evaluate_objectives(solution):
        selected = np.where(solution == 1)[0]
        coverage = biased_importance[selected].sum() if len(selected) > 0 else 0

        if len(selected) > 1:
            pairs = adj_matrix[np.ix_(selected, selected)]
            mask = pairs > 0.2
            np.fill_diagonal(mask, False)
            redundancy = pairs[mask].mean() if mask.any() else 0
        else:
            redundancy = 0

        length_penalty = abs(len(selected) - target_length)
        return coverage, redundancy, length_penalty

    def fitness(solution):
        cov, red, lpen = evaluate_objectives(solution)
        return cov - red - 2 * (lpen ** 2)

    def repair_solution(solution):
        sol = solution.copy()
        selected = np.where(sol == 1)[0]
        while len(selected) > target_length:
            importances = biased_importance[selected]
            drop_idx = selected[importances.argmin()]
            sol[drop_idx] = 0
            selected = np.where(sol == 1)[0]
        while len(selected) < target_length:
            unselected = np.where(sol == 0)[0]
            if len(unselected) == 0:
                break
            importances = biased_importance[unselected]
            add_idx = unselected[importances.argmax()]
            sol[add_idx] = 1
            selected = np.where(sol == 1)[0]
        return sol

    def initialize_population():
        population = []
        probs = biased_importance / biased_importance.sum() \
            if biased_importance.sum() > 0 else np.ones(num_sentences) / num_sentences
        probs = np.clip(probs * 5, 0, 1)
        for _ in range(pop_size):
            sol = (np.random.random(num_sentences) < probs).astype(int)
            sol = repair_solution(sol)
            population.append(sol)
        return np.array(population)

    def sigmoid(x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def update_position_woa(current, leader, a, population):
        new_solution = current.copy()
        r1 = np.random.random()
        r2 = np.random.random()
        A = 2 * a * r1 - a
        C = 2 * r2
        p = np.random.random()
        b = 1

        if p < 0.5:
            if abs(A) < 1:
                D = np.abs(C * leader - current)
                transfer = sigmoid(leader - A * D)
                new_solution = (np.random.random(num_sentences) < transfer).astype(int)
            else:
                rand_idx = np.random.randint(0, pop_size)
                rand_sol = population[rand_idx]
                D = np.abs(C * rand_sol - current)
                transfer = sigmoid(rand_sol - A * D)
                new_solution = (np.random.random(num_sentences) < transfer).astype(int)
        else:
            D_prime = np.abs(leader - current)
            l_param = np.random.uniform(-1, 1)
            spiral = D_prime * np.exp(b * l_param) * np.cos(2 * np.pi * l_param)
            transfer = sigmoid(leader + spiral)
            new_solution = (np.random.random(num_sentences) < transfer).astype(int)

        return repair_solution(new_solution)

    population = initialize_population()

    for t in range(max_iter):
        a = 2 - 2 * t / max_iter
        fitnesses = np.array([fitness(sol) for sol in population])
        leader_idx = fitnesses.argmax()
        leader = population[leader_idx].copy()

        for i in range(pop_size):
            population[i] = update_position_woa(
                population[i], leader, a, population
            )

    fitnesses = np.array([fitness(sol) for sol in population])
    best_idx = fitnesses.argmax()
    best_solution = population[best_idx]

    return best_solution

def graph_from_cache(c):
    G = nx.Graph()
    for i in range(c.n):
        G.add_node(i, importance=float(c.node_importance[i]))
    iu, ju = np.triu_indices(c.n, k=1)
    for i, j in zip(iu, ju):
        if c.adj[i, j] > 0:
            G.add_edge(int(i), int(j), weight=float(c.adj[i, j]))
    return G

np.random.seed(42)            # same seed as [eval_v1_4_1] CELL 1
SOWOA_IDX, t0, adj_ok, pb_ok = [], time.time(), 0, 0
for a in AIDS:
    c = cache_by_id[a]
    pb_ok += np.allclose(c.position_bonus, mo.build_position_bonus(c.n))
    G = graph_from_cache(c)
    A = nx.to_numpy_array(G, weight='weight')
    off = ~np.eye(c.n, dtype=bool)                 # the diagonal is never used
    adj_ok += np.allclose(A[off], c.adj[off])
    best = single_objective_woa(G, pop_size=30, max_iter=50, summary_size=SUMMARY_SIZE,
                                position_bonus=c.position_bonus)
    SOWOA_IDX.append(tuple(int(i) for i in np.where(best == 1)[0]))
SUMM['WOA satu objektif'] = [' '.join(SENTS[a][i] for i in idx) for a, idx in zip(AIDS, SOWOA_IDX)]
out(f'[W3] SO-WOA done in {time.time() - t0:.0f}s (np.random.seed(42), cache inputs); '
    f'graph == cached adjacency {adj_ok}/150, position bonus == cell-28 schedule {pb_ok}/150')

## [W4] IndoBERT inference with the cached fine-tuned model (eval CELL 10)

In [ ]:
INDOBERT_STATUS = 'not run'
model_path  = f'{INDOBERT_DIR}/indobert_extractor.pt'
labels_path = f'{INDOBERT_DIR}/indobert_oracle_labels.pkl'
if os.path.exists(model_path):
    import torch, torch.nn as nn
    from transformers import AutoTokenizer, AutoModel
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    MODEL_NAME = 'indobenchmark/indobert-base-p2'
    MAX_SENT_LEN, INFERENCE_BATCH = 128, 64

    class IndoBERTExtractor(nn.Module):          # verbatim
        def __init__(self, model_name):
            super().__init__()
            self.bert = AutoModel.from_pretrained(model_name)
            self.classifier = nn.Linear(self.bert.config.hidden_size, 1)
            self.dropout = nn.Dropout(0.1)
        def forward(self, input_ids, attention_mask):
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            cls_output = self.dropout(outputs.last_hidden_state[:, 0, :])
            return self.classifier(cls_output).squeeze(-1)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.model_max_length > 1_000_000:
        tokenizer.model_max_length = 512
    model = IndoBERTExtractor(MODEL_NAME).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))

    def score_sentences_batched(sentences, model, tok, batch_size=64, max_length=128):  # verbatim
        if len(sentences) == 0:
            return []
        all_scores = []
        model.eval()
        with torch.no_grad():
            for i in range(0, len(sentences), batch_size):
                batch_sents = sentences[i:i+batch_size]
                enc = tok(batch_sents, truncation=True, max_length=max_length,
                          padding=True, return_tensors='pt')
                logits = model(enc['input_ids'].to(device), enc['attention_mask'].to(device))
                all_scores.extend(torch.sigmoid(logits).cpu().numpy().tolist())
        return all_scores

    ib = []
    for a in AIDS:
        sentences = SENTS[a]
        if len(sentences) <= SUMMARY_SIZE:
            ib.append(' '.join(sentences)); continue
        scores = score_sentences_batched(sentences, model, tokenizer,
                                         batch_size=INFERENCE_BATCH, max_length=MAX_SENT_LEN)
        ranked = np.argsort(scores)[::-1]
        selected_indices = sorted(ranked[:SUMMARY_SIZE].tolist())
        ib.append(' '.join([sentences[i] for i in selected_indices]))
    SUMM['IndoBERT'] = ib
    INDOBERT_STATUS = f'cached model, {device}'
    out(f'\n[W4] IndoBERT inference done ({INDOBERT_STATUS}); checkpoint {model_path}')
    ib_rx = []                                    # same model on the old regex units
    for a in AIDS:
        s_ = REGEX[a]
        if len(s_) <= SUMMARY_SIZE:
            ib_rx.append(' '.join(s_)); continue
        sco = score_sentences_batched(s_, model, tokenizer, batch_size=INFERENCE_BATCH, max_length=MAX_SENT_LEN)
        ib_rx.append(' '.join(s_[i] for i in sorted(np.argsort(sco)[::-1][:SUMMARY_SIZE].tolist())))
    out(f'  cached model on regex units: R-1 {r1(ib_rx):.4f} (Tabel 4.18 now: 0.4660) '
        f'-> {"same model as the reported run" if abs(r1(ib_rx) - 0.4660) < 5e-5 else "DIFFERS from the reported run"}')
else:
    out(f'\n[W4] IndoBERT model NOT FOUND at {model_path} → IndoBERT row PENDING')

# Q9: which articles trained the cached model? (oracle-label cache, if present)
if os.path.exists(labels_path):
    with open(labels_path, 'rb') as f:
        tr_sents, tr_labels = pickle.load(f)
    tr_text = [' '.join(s) for s in tr_sents]
    train_text = {' '.join(split_text_into_sentences(p)): i
                  for i, p in enumerate(train_full.paragraphs[:3000])}
    pos = [train_text.get(t) for t in tr_text]
    found = [p for p in pos if p is not None]
    eval_text = {' '.join(SENTS[a]) for a in AIDS}
    out(f'[W4/Q9] oracle-label cache: {len(tr_sents)} training articles; located in train.01 '
        f'(first 3000): {len(found)}; index range {min(found) if found else None}–'
        f'{max(found) if found else None}; overlap with the evaluated 150: '
        f'{sum(t in eval_text for t in tr_text)}')
else:
    out(f'[W4/Q9] oracle-label cache not found at {labels_path}')

## [W5] MO-WOA+ (FROZEN) rerun from the cache, keeping the archive

In [ ]:
# run_mowoa_final_arch is run_mowoa_final with one change: it also returns the final
# archive (knee_B, knee_F), so every selection weight can be applied to the same
# archive. search, budget and rng are not touched
_src = inspect.getsource(mo.run_mowoa_final)
_src = _src.replace('def run_mowoa_final(', 'def run_mowoa_final_arch(', 1)
_old = 'knee_row=knee_row, eval_rows=log.rows)'
assert _src.count(_old) == 1
_src = _src.replace(_old, 'knee_row=knee_row, eval_rows=log.rows, '
                          'knee_B=np.asarray(knee_B), knee_F=np.asarray(knee_F))')
exec(compile(_src, 'run_mowoa_final_arch', 'exec'), mo.__dict__)
run_arch = mo.run_mowoa_final_arch

def select_row(knee_B, knee_F, enum, w):
    """Selection rule of run_mowoa_final / cell 21 l.432-442 with weights w."""
    native = np.column_stack([-knee_F[:, 0], knee_F[:, 1]])
    omin, omax = native.min(axis=0), native.max(axis=0)
    rngv = omax - omin
    rngv[rngv == 0] = 1
    norm = (native - omin) / rngv
    norm[:, 0] = 1 - norm[:, 0]
    d = np.sqrt((np.asarray(w, float) * norm ** 2).sum(axis=1))
    return enum.row_of(knee_B[int(d.argmin())])

t_start = time.time()
for k, aid in enumerate(AIDS):
    ck = f'{MP_DIR}/article_{aid:03d}.pkl'
    rows = pickle.load(open(ck, 'rb')) if os.path.exists(ck) else []
    todo = [s for s in SEEDS if s not in {r['seed'] for r in rows}]   # resume: only missing seeds
    if not todo:
        continue
    e = enums[aid]
    for seed in todo:
        t0 = time.time()
        r = run_arch(e, seed, **FROZEN)
        assert r['n_eval'] == mo.FE_BUDGET
        rec = dict(article=aid, seed=seed, sec=time.time() - t0, n_eval=r['n_eval'],
                   n_unique=len(set(r['eval_rows'])), knee_row_module=r['knee_row'],
                   arch_rows=[e.row_of(b) for b in r['knee_B']])
        for name, w in WEIGHTS.items():
            rec[f'row_{name}'] = select_row(r['knee_B'], r['knee_F'], e, w)
        rows.append(rec)
    with open(ck, 'wb') as f:
        pickle.dump(rows, f)
    if k % 10 == 0:
        el = time.time() - t_start
        print(f'  article {aid:>3} (N={e.cache.n}) done; elapsed {el/60:.1f} min')

MP = pd.DataFrame([r for aid in AIDS for r in pickle.load(open(f'{MP_DIR}/article_{aid:03d}.pkl', 'rb'))])
MP = MP[MP.seed.isin(SEEDS)].sort_values(['article', 'seed']).reset_index(drop=True)
assert len(MP) == len(AIDS) * len(SEEDS)
out(f'\n[W5] MO-WOA+ reruns: {len(MP)} runs ({len(SEEDS)} seeds), mean {MP.sec.mean():.2f}s/run')
out('reproduction checks (w = [32,1]):')
out(f'  selection rule here == run_mowoa_final knee_row: {(MP.row_w32 == MP.knee_row_module).sum()}/{len(MP)}')
pc = [(knees.get((r.article, 'MO-WOA+', r.seed)) == r.row_w32) for r in MP.itertuples()]
out(f'  rerun == Phase C checkpoint knee_row           : {sum(pc)}/{len(MP)}')
uq = MP.n_unique
out(f'  unique candidates per run (of 1480): mean {uq.mean():.1f}, min {uq.min()}, max {uq.max()}; '
    f'hard articles mean {MP[MP.article.map(lambda a: enums[a].M > mo.FE_BUDGET)].n_unique.mean():.1f}')

## [W6] scoring, Tabel 4.18 (native), paired intervals, seed sensitivity, S9

In [ ]:
_rc = {}
def score_text(ref, text):
    s = sc.score(ref, text)
    return (s['rouge1'].fmeasure, s['rouge2'].fmeasure, s['rougeL'].fmeasure,
            s['rouge1'].precision, s['rouge1'].recall,
            len(text.split()), len(rs_tokenize.tokenize(text, None)))
def score_row(aid, row):
    key = (aid, int(row))
    if key not in _rc:
        combo = enums[aid].combos[int(row)]
        _rc[key] = score_text(golden[aid], ' '.join(SENTS[aid][i] for i in sorted(combo.tolist())))
    return _rc[key]
COLS = ['r1', 'r2', 'rl', 'r1p', 'r1r', 'words', 'tokens']

PER = {}   # algorithm -> (150 x 7) array, article order = AIDS
for name, texts in SUMM.items():
    PER[name] = np.array([score_text(golden[a], t) for a, t in zip(AIDS, texts)])
lead_rows = {a: enums[a].index_of[LEAD3] for a in AIDS}
for name in WEIGHTS:
    sc_arr = np.array([score_row(r.article, getattr(r, f'row_{name}')) for r in MP.itertuples()])
    for j, c in enumerate(COLS):
        MP[f'{c}_{name}'] = sc_arr[:, j]
    MP[f'lead3_{name}'] = [getattr(r, f'row_{name}') == lead_rows[r.article] for r in MP.itertuples()]

def seed_mean(name):    # per-article mean over seeds -> (150 x 7)
    g = MP.groupby('article')[[f'{c}_{name}' for c in COLS]].mean().loc[AIDS]
    return g.to_numpy()
def seed0(name):
    g = MP[MP.seed == SEEDS[0]].set_index('article').loc[AIDS][[f'{c}_{name}' for c in COLS]]
    return g.to_numpy()
PER['MO-WOA+ [1,1] (rata-rata seed)'] = seed_mean('w1')
PER['MO-WOA+ [1,1] seed 0']           = seed0('w1')
PER['MO-WOA+ [32,1] (rata-rata seed)'] = seed_mean('w32')

ORDER = ['MO-WOA+ [1,1] (rata-rata seed)', 'LEAD-3', 'WOA satu objektif', 'IndoBERT', 'TextRank', 'LexRank']
fmt = lambda v: f'{v:.4f}'.replace('.', ',')
out('\n' + '=' * 78)
out('TABEL 4.18 (native segmentation; statistics over 150 per-article F1; SD ddof=1)')
out('Metrik\tAlgoritma\tRata-rata\tSimpangan baku\tMin\tMaks\tKata (rata-rata)')
for j, m in enumerate(['ROUGE-1 F1', 'ROUGE-2 F1', 'ROUGE-L F1']):
    for name in ORDER + ['MO-WOA+ [1,1] seed 0', 'MO-WOA+ [32,1] (rata-rata seed)']:
        if name not in PER:
            out(f'{m}\t{name}\tPENDING'); continue
        x = PER[name][:, j]
        wd = f'{PER[name][:, 5].mean():.1f}'.replace('.', ',')
        out(f'{m}\t{name}\t{fmt(x.mean())}\t{fmt(x.std(ddof=1))}\t{fmt(x.min())}\t{fmt(x.max())}\t{wd}')
out('\nROUGE-1 precision / recall and length per algorithm (means over 150):')
for name in ORDER:
    if name in PER:
        P = PER[name]
        out(f'  {name:<32} P {P[:,3].mean():.4f}  R {P[:,4].mean():.4f}  '
            f'words {P[:,5].mean():.2f}  ROUGE-tokens {P[:,6].mean():.2f}')
lead_txt = SUMM['LEAD-3']
for name in ('TextRank', 'LexRank', 'WOA satu objektif', 'IndoBERT'):
    if name in SUMM:
        out(f'  {name:<32} identical to native LEAD-3 in '
            f'{sum(a.strip() == b.strip() for a, b in zip(SUMM[name], lead_txt))}/150')

def paired(a, b, label):
    d = a - b; n = len(d)
    m, se = d.mean(), d.std(ddof=1) / np.sqrt(n)
    t = stats.t.ppf(0.975, n - 1)
    rng = np.random.default_rng(0)
    boot = np.array([d[rng.integers(0, n, n)].mean() for _ in range(10000)])
    out(f'  {label:<44} {m:+.4f}  t [{m - t*se:+.4f}, {m + t*se:+.4f}]  '
        f'boot [{np.percentile(boot, 2.5):+.4f}, {np.percentile(boot, 97.5):+.4f}]  '
        f'higher/equal/lower {int((d > 1e-12).sum())}/{int((abs(d) <= 1e-12).sum())}/{int((d < -1e-12).sum())}')
out('\nPAIRED ARTICLE-LEVEL DIFFERENCES vs native LEAD-3 (descriptive 95% intervals, n = 150)')
for name in ORDER[:1] + ORDER[2:] + ['MO-WOA+ [1,1] seed 0', 'MO-WOA+ [32,1] (rata-rata seed)']:
    if name not in PER:
        continue
    for j, m in enumerate(['R-1', 'R-2', 'R-L']):
        paired(PER[name][:, j], PER['LEAD-3'][:, j], f'{name} {m}')
    paired(PER[name][:, 5], PER['LEAD-3'][:, 5], f'{name} words')

out('\nSEED SENSITIVITY (corpus means per seed; n seeds = %d)' % len(SEEDS))
out('weight\tR-1 mean±SD [min,max]\tR-2 mean±SD\tR-L mean±SD\tseed-0 R-1\tarticles = LEAD-3 per seed (mean [min,max])')
for name in WEIGHTS:
    g = MP.groupby('seed')[[f'r1_{name}', f'r2_{name}', f'rl_{name}', f'lead3_{name}']]
    gm = g.mean(); gl = MP.groupby('seed')[f'lead3_{name}'].sum()
    out(f'{name}\t{gm[f"r1_{name}"].mean():.4f}±{gm[f"r1_{name}"].std(ddof=1):.4f} '
        f'[{gm[f"r1_{name}"].min():.4f},{gm[f"r1_{name}"].max():.4f}]\t'
        f'{gm[f"r2_{name}"].mean():.4f}±{gm[f"r2_{name}"].std(ddof=1):.4f}\t'
        f'{gm[f"rl_{name}"].mean():.4f}±{gm[f"rl_{name}"].std(ddof=1):.4f}\t'
        f'{gm.loc[SEEDS[0], f"r1_{name}"]:.4f}\t{gl.mean():.1f} [{gl.min()},{gl.max()}]')
frac = MP.groupby('article')['lead3_w1'].mean().loc[AIDS]
out(f'[1,1]: articles = LEAD-3 in all seeds {int((frac == 1).sum())}, in no seed {int((frac == 0).sum())}, '
    f'in some seeds {int(((frac > 0) & (frac < 1)).sum())}')

out('\nS9 / Q22 CHECKS')
greedy = {a: tuple(sorted(np.argsort(-cache_by_id[a].biased_importance)[:3].tolist())) for a in AIDS}
out(f'greedy top-3 by biased importance == LEAD-3: {sum(greedy[a] == LEAD3 for a in AIDS)}/150; '
    f'coverage extreme == greedy: {sum(tuple(enums[a].combos[enums[a].true_cov_extreme_idx]) == greedy[a] for a in AIDS)}/150')
s0 = MP[MP.seed == SEEDS[0]]
off_w32 = [int(r.article) + 1 for r in s0.itertuples() if not r.lead3_w32]
out(f'seed {SEEDS[0]}, w = [32,1]: articles NOT = LEAD-3 (1-based): {off_w32}')
gen = canon['generated_summaries'][:N_ARTICLES]          # the notebook run behind old Tabel 4.18
s0_txt = {int(r.article): ' '.join(SENTS[int(r.article)][i] for i in sorted(enums[int(r.article)].combos[int(r.row_w32)].tolist()))
          for r in s0.itertuples()}
out(f'canonical JSON (notebook run, w = [32,1]) == native LEAD-3 text: '
    f'{sum(gen[a].strip() == SUMM["LEAD-3"][k].strip() for k, a in enumerate(AIDS))}/150; '
    f'== cache-pipeline seed {SEEDS[0]} w = [32,1]: {sum(gen[a].strip() == s0_txt[a].strip() for a in AIDS)}/150')
ever = MP[~MP.lead3_w32].groupby('article').size()
out('all seeds, w = [32,1]: article (1-based) -> number of seeds not = LEAD-3:',
    {int(a) + 1: int(n) for a, n in ever.items()})
c148 = cache_by_id[148]
out('Artikel 149 (idx 148), cache: LexRank s0..s2', np.round(c148.node_importance[:3], 6).tolist(),
    '| biased', np.round(c148.biased_importance[:3], 6).tolist(),
    f'| |P*| {enums[148].m} | max coverage {enums[148].F_native[:,0].max():.6f}')
out('   notebook cell 43 printed: LexRank [0.033026, 0.035520, 0.034571], biased '
    '[0.099077, 0.088800, 0.086427], |P*| 13, max coverage 0.274304')
r149 = MP[(MP.article == 148) & (MP.seed == SEEDS[0])].iloc[0]
for name in ('w1', 'w32'):
    row = int(getattr(r149, f'row_{name}'))
    out(f'   Artikel 149 seed {SEEDS[0]} {name}: sentences {enums[148].combos[row].tolist()} '
        f'objectives (cov, red) {np.round(enums[148].F_native[row], 6).tolist()} '
        f'R-1/2/L {np.round(score_row(148, row)[:3], 4).tolist()}')

## [W7] save

In [ ]:
def r4(x): return np.round(x, 4)
pa = pd.DataFrame({'article': np.array(AIDS) + 1, 'N': [enums[a].cache.n for a in AIDS]})
short = {'LEAD-3': 'lead3', 'TextRank': 'textrank', 'LexRank': 'lexrank', 'WOA satu objektif': 'sowoa',
         'IndoBERT': 'indobert', 'MO-WOA+ [1,1] (rata-rata seed)': 'mwp_eq',
         'MO-WOA+ [1,1] seed 0': 'mwp_eq_s0', 'MO-WOA+ [32,1] (rata-rata seed)': 'mwp_w32'}
for name, s in short.items():
    if name in PER:
        for j, c in enumerate(['r1', 'r2', 'rl']):
            pa[f'{s}_{c}'] = r4(PER[name][:, j])
        pa[f'{s}_wd'] = np.round(PER[name][:, 5], 1)
pa['mwp_eq_lead3_frac'] = np.round(frac.to_numpy(), 3)
pa.to_csv(f'{WP1_DIR}/wp1_per_article.csv', index=False)
keep = ['article', 'seed', 'sec', 'n_unique'] + [f'row_{n}' for n in WEIGHTS] + \
       [f'{c}_{n}' for n in ('w1', 'w32') for c in ('r1', 'r2', 'rl', 'words')]
MP[keep].assign(article=MP.article + 1).to_csv(f'{WP1_DIR}/wp1_mowoa_plus_runs.csv', index=False)
with open(f'{WP1_DIR}/wp1_summary.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(LOG))
for fn in sorted(os.listdir(WP1_DIR)):
    fp = f'{WP1_DIR}/{fn}'
    if os.path.isfile(fp):
        print(f'{fn:<28} {os.path.getsize(fp)/1024:7.1f} KB')
print('\nfiles to keep: wp1_summary.txt, wp1_per_article.csv')